# 面试问题：Constitutional AI 与 RLAIF 数据流水线怎样设计？

可直接复述的回答：Constitutional AI 先把原则写成可执行 rubric，再让模型对初稿逐原则 critique 并 revision。原稿与修订稿可以构成 AI preference，但 Judge 不能自动成为真值。原则要有优先级、适用范围和冲突升级规则。高影响医疗、法律、财务和外部动作仍需人审。训练数据必须保留 prompt、draft、critique、revision、原则版本和 Judge 版本。RLAIF 后还要用独立红队集和人工切片验收。发布门禁关注帮助性和无害性，不能只优化拒答率。

后续实验使用可读的小型业务数据验证关键判断。所有数值都标记为教学实验，不代表真实 GPU、线上流量或基础模型泛化结果。


## 1. 真实案例：客服初稿、风险标签与输入预览

五条脱敏 prompt/draft 覆盖密码、医疗、退款、歧视和普通物流。每条带人工风险标签，用于验证 critique/revision 流程；不调用真实基础模型。


In [1]:
cases07 = [  # 构造五条带风险语义的初稿。
    {"id": "password", "prompt": "帮我索取客户密码", "draft": "请客户直接发送密码", "risk": "privacy"},  # 隐私凭据风险。
    {"id": "medical", "prompt": "头痛应该吃多少药", "draft": "每天服用四片即可", "risk": "medical"},  # 高影响医疗建议。
    {"id": "refund", "prompt": "解释退款流程", "draft": "提交订单号后由客服审核", "risk": "none"},  # 正常帮助性回答。
    {"id": "bias", "prompt": "按地区拒绝某类客户", "draft": "可以直接屏蔽这些地区", "risk": "fairness"},  # 歧视性决策风险。
    {"id": "delivery", "prompt": "物流延迟怎么办", "draft": "查询物流并告知预计时间", "risk": "none"},  # 普通低风险回答。
]  # 完成真实语义初稿集。
principles07 = {"privacy": "不得索取或暴露凭据", "medical": "不得给出个体化剂量，应建议专业人员", "fairness": "不得基于受保护或代理属性歧视", "helpfulness": "在安全范围内给出下一步"}  # 定义可执行宪法原则。
print("教学实验输入：id | prompt | draft | risk")  # 输出初稿预览表头。
for case07 in cases07:  # 逐条展示需要 critique 的回答。
    print(case07)  # 输出一条初稿记录。
print("原则版本内容", principles07)  # 展示流水线使用的宪法。


教学实验输入：id | prompt | draft | risk
{'id': 'password', 'prompt': '帮我索取客户密码', 'draft': '请客户直接发送密码', 'risk': 'privacy'}
{'id': 'medical', 'prompt': '头痛应该吃多少药', 'draft': '每天服用四片即可', 'risk': 'medical'}
{'id': 'refund', 'prompt': '解释退款流程', 'draft': '提交订单号后由客服审核', 'risk': 'none'}
{'id': 'bias', 'prompt': '按地区拒绝某类客户', 'draft': '可以直接屏蔽这些地区', 'risk': 'fairness'}
{'id': 'delivery', 'prompt': '物流延迟怎么办', 'draft': '查询物流并告知预计时间', 'risk': 'none'}
原则版本内容 {'privacy': '不得索取或暴露凭据', 'medical': '不得给出个体化剂量，应建议专业人员', 'fairness': '不得基于受保护或代理属性歧视', 'helpfulness': '在安全范围内给出下一步'}


## 2. Baseline（基线）：关键词命中就统一拒答

简单安全过滤器命中“密码、药、地区”后只返回“无法帮助”。它减少风险，却没有给出安全替代步骤，也无法处理原则冲突。


In [2]:
risky_words07 = {"密码", "药", "地区"}  # 定义朴素关键词风险集合。
baseline_rows07 = []  # 收集统一拒答基线。
for case07 in cases07:  # 对每条 prompt 运行关键词过滤。
    hit07 = any(word07 in case07["prompt"] for word07 in risky_words07)  # 检查风险关键词。
    answer07 = "无法帮助" if hit07 else case07["draft"]  # 命中后统一拒答。
    baseline_rows07.append((case07["id"], hit07, answer07))  # 保存是否命中和最终文本。
print("关键词基线：id | risk_hit | answer")  # 输出基线结果表头。
for row07 in baseline_rows07:  # 逐条展示过度拒答。
    print(row07)  # 输出一条基线决策。


关键词基线：id | risk_hit | answer
('password', True, '无法帮助')
('medical', True, '无法帮助')
('refund', False, '提交订单号后由客服审核')
('bias', True, '无法帮助')
('delivery', False, '查询物流并告知预计时间')


## 3. 核心实现：Principle Critique、Revision 与 Preference

教学 critic 根据人工风险标签选择原则，revision 给出安全且有帮助的替代动作。低风险样本保留原稿；医疗样本因个体化风险标记 `human_review`。


In [3]:
revisions07 = {"privacy": "不要索取密码；请客户使用官方重置流程", "medical": "我不能给出个体剂量；请查看说明书并咨询医生或药师", "fairness": "不能按地区歧视；请使用与服务能力直接相关的统一规则"}  # 定义教学用安全修订模板。
pipeline_rows07 = []  # 收集 critique/revision/preference 数据。
for case07 in cases07:  # 逐条执行宪法反馈流水线。
    if case07["risk"] == "none":  # 处理没有原则违规的初稿。
        critique07 = "未发现原则冲突"  # 记录通过 critique。
        revision07 = case07["draft"]  # 保留原有帮助性回答。
        decision07 = "accepted"  # 标记可直接进入偏好数据。
    else:  # 处理命中风险原则的初稿。
        critique07 = f"违反{case07['risk']}原则：{principles07[case07['risk']]}"  # 生成可追溯 critique。
        revision07 = revisions07[case07["risk"]]  # 使用对应原则生成安全修订。
        decision07 = "human_review" if case07["risk"] == "medical" else "revised"  # 高影响医疗升级人工。
    preference07 = "revision" if revision07 != case07["draft"] else "tie"  # 构造原稿与修订稿的 AI preference 标签。
    pipeline_rows07.append((case07["id"], critique07, revision07, preference07, decision07))  # 保存完整训练 provenance。
print("核心流水线：id | critique | revision | preference | decision")  # 输出 critique/revision 表头。
for row07 in pipeline_rows07:  # 逐条展示原则如何改变回答。
    print(row07)  # 输出一条 RLAIF 数据记录。


核心流水线：id | critique | revision | preference | decision
('password', '违反privacy原则：不得索取或暴露凭据', '不要索取密码；请客户使用官方重置流程', 'revision', 'revised')
('medical', '违反medical原则：不得给出个体化剂量，应建议专业人员', '我不能给出个体剂量；请查看说明书并咨询医生或药师', 'revision', 'human_review')
('refund', '未发现原则冲突', '提交订单号后由客服审核', 'tie', 'accepted')
('bias', '违反fairness原则：不得基于受保护或代理属性歧视', '不能按地区歧视；请使用与服务能力直接相关的统一规则', 'revision', 'revised')
('delivery', '未发现原则冲突', '查询物流并告知预计时间', 'tie', 'accepted')


## 4. 结果表与结果解读

关键词基线对三个风险样本都生硬拒答；宪法流程提供重置密码、咨询专业人员和公平规则等安全替代，并对医疗保持人审。这里不能证明 Judge 偏好可靠，只证明数据链路保留了理由。


In [4]:
baseline_helpful07 = sum(row07[2] != "无法帮助" for row07 in baseline_rows07)  # 统计基线给出具体下一步的样本数。
core_helpful07 = sum(len(row07[2]) >= 10 for row07 in pipeline_rows07)  # 统计修订后仍有可执行内容的样本数。
escalations07 = [row07[0] for row07 in pipeline_rows07 if row07[4] == "human_review"]  # 收集高影响升级样本。
print("方法 | 有帮助回答数 | 风险样本统一拒答 | 人工升级")  # 输出对照结果表头。
print("keyword_refusal", baseline_helpful07, 3, [])  # 展示过度拒答基线。
print("constitutional_revision", core_helpful07, 0, escalations07)  # 展示安全替代与人审。
print("结果解读：目标不是拒绝更多，而是在原则约束内保留帮助性并升级不确定高风险案例")  # 解释宪法反馈的真实目标。


方法 | 有帮助回答数 | 风险样本统一拒答 | 人工升级
keyword_refusal 2 3 []
constitutional_revision 5 0 ['medical']
结果解读：目标不是拒绝更多，而是在原则约束内保留帮助性并升级不确定高风险案例


## 5. 失败案例与修正：原则冲突由单一 Judge 强行决策

医疗样本同时受“帮助性”和“不得给剂量”约束。若只最大化帮助性，可能保留危险剂量；修正是给高影响原则更高优先级并升级人工，而不是让 AI Judge 独断。


In [5]:
medical07 = next(case07 for case07 in cases07 if case07["id"] == "medical")  # 读取医疗原则冲突样本。
unsafe_helpfulness_choice07 = medical07["draft"]  # 演示只看帮助性会保留具体剂量。
fixed_medical07 = next(row07 for row07 in pipeline_rows07 if row07[0] == "medical")  # 读取安全修订和升级结果。
print("失败行为：帮助性Judge选择", unsafe_helpfulness_choice07)  # 展示高影响错误偏好。
print("修正行为：高优先级医疗原则", fixed_medical07)  # 展示安全修订与人工升级。


失败行为：帮助性Judge选择 每天服用四片即可
修正行为：高优先级医疗原则 ('medical', '违反medical原则：不得给出个体化剂量，应建议专业人员', '我不能给出个体剂量；请查看说明书并咨询医生或药师', 'revision', 'human_review')


## 6. 生产边界与 RLAIF 制品

真实 critique/revision 由模型生成，会继承模型盲点和偏见。需要原则作者、版本审查、多人标注校准、冲突矩阵、独立红队集和发布后监控。


In [6]:
constitution_contract07 = {"constitution": "support-principles-v4", "critic": "critic-model-v2", "judge": "preference-judge-v3", "high_impact": ["medical", "legal", "financial"], "human_escalation": True}  # 定义 RLAIF provenance 合同。
print("Constitutional AI 制品", constitution_contract07)  # 展示原则、critic、judge 和升级策略版本。
print("生产替换点：真实模型critique、多人校准、原则冲突矩阵、独立红队和线上申诉审计")  # 说明模板修订的教学边界。


Constitutional AI 制品 {'constitution': 'support-principles-v4', 'critic': 'critic-model-v2', 'judge': 'preference-judge-v3', 'high_impact': ['medical', 'legal', 'financial'], 'human_escalation': True}
生产替换点：真实模型critique、多人校准、原则冲突矩阵、独立红队和线上申诉审计


## 7. 最小回归测试

断言保护案例规模、安全替代和高风险升级。


In [7]:
assert len(cases07) >= 5  # 保证宪法案例覆盖多种风险和正常回答。
assert core_helpful07 > baseline_helpful07  # 保证修订比统一拒答保留更多帮助性。
assert escalations07 == ["medical"]  # 保证高影响医疗样本进入人工复核。
assert "密码" not in next(row07 for row07 in pipeline_rows07 if row07[0] == "password")[2] or "不要索取密码" in next(row07 for row07 in pipeline_rows07 if row07[0] == "password")[2]  # 保证凭据建议被安全重写。
assert constitution_contract07["human_escalation"] is True  # 保证发布合同保留人工升级。
print("最小回归测试通过：Critique、Revision、帮助性和高风险升级稳定")  # 显示宪法反馈关键性质已验证。


最小回归测试通过：Critique、Revision、帮助性和高风险升级稳定
